# TP 4 : IA & Santé - Deep Learning 🧠
(CORRECTION DÉTAILLÉE)

**Objectif :** Configurer un MLP avec Early Stopping.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, 
                             auc, roc_auc_score, accuracy_score, precision_score, 
                             recall_score, f1_score)
import seaborn as sns

# Charger les données
data = load_breast_cancer()
X, y = data.data, data.target

# Séparation appropriée: Train (70%), Val (15%), Test (15%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.176, random_state=42  # 176% de 70% ≈ 15%
)

# Normalisation (IMPORTANT pour les réseaux de neurones)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Dimensions des données:")
print(f"  Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}, Val: {X_val_scaled.shape}")
print(f"  Classes: 0={np.sum(y_train==0)}, 1={np.sum(y_train==1)} (Train)")

## Entraînement avec Early Stopping

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(50, 30), 
    max_iter=1000, 
    early_stopping=True, 
    validation_fraction=0.2,
    learning_rate_init=0.001,
    random_state=42,
    verbose=1
)
mlp.fit(X_train_scaled, y_train)

print(f"\nModèle entraîné !")
print(f"  Couches: Input={X_train_scaled.shape[1]} → {mlp.hidden_layer_sizes} → Output=1")
print(f"  Nombre d'itérations: {mlp.n_iter_}")
print(f"  Perte finale: {mlp.loss_:.4f}")

## Analyse de l'apprentissage

In [ ]:
# Visualiser les courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de perte
axes[0].plot(mlp.loss_curve_, linewidth=2, label='Train Loss', marker='o', markersize=3, alpha=0.7)
axes[0].set_xlabel('Époque', fontsize=12)
axes[0].set_ylabel('Perte (Loss)', fontsize=12)
axes[0].set_title('Courbe de Perte - Early Stopping', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Score de validation
axes[1].plot(mlp.validation_scores_, linewidth=2, label='Validation Score', 
             marker='s', markersize=3, color='green', alpha=0.7)
axes[1].set_xlabel('Époque', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Score de Validation - Early Stopping', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"✓ Perte initiale: {mlp.loss_curve_[0]:.4f}")
print(f"✓ Perte finale: {mlp.loss_curve_[-1]:.4f}")
print(f"✓ Meilleur score validation: {max(mlp.validation_scores_):.4f}")

## Évaluation sur l'ensemble de Test

In [ ]:
# Prédictions sur l'ensemble de test
y_pred = mlp.predict(X_test_scaled)
y_pred_proba = mlp.predict_proba(X_test_scaled)[:, 1]  # Probabilités de classe 1

# Calculer les métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MÉTRIQUES DE PERFORMANCE SUR LE TEST SET")
print("=" * 50)
print(f"✓ Accuracy (Exactitude):   {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"✓ Precision (Précision):   {precision:.4f}")
print(f"✓ Recall (Rappel):         {recall:.4f}")
print(f"✓ F1-Score:                {f1:.4f}")
print(f"✓ ROC-AUC:                 {roc_auc:.4f}")
print("=" * 50)

# Rapport de classification détaillé
print("\nRAPPORT DE CLASSIFICATION:")
print(classification_report(y_test, y_pred, target_names=['Bénin (0)', 'Malin (1)']))

## Matrice de Confusion et Visualisations

In [ ]:
# Calculer la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Créer une figure avec plusieurs visualisations
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Matrice de confusion
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0, 0],
            xticklabels=['Bénin', 'Malin'], yticklabels=['Bénin', 'Malin'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0, 0].set_xlabel('Prédiction', fontsize=12)
axes[0, 0].set_ylabel('Réalité', fontsize=12)
axes[0, 0].set_title('Matrice de Confusion', fontsize=13, fontweight='bold')

# 2. Distribution des probabilités prédites
axes[0, 1].hist(y_pred_proba[y_test == 0], bins=20, alpha=0.7, label='Bénin (0)', color='blue')
axes[0, 1].hist(y_pred_proba[y_test == 1], bins=20, alpha=0.7, label='Malin (1)', color='red')
axes[0, 1].axvline(0.5, color='black', linestyle='--', linewidth=2, label='Seuil (0.5)')
axes[0, 1].set_xlabel('Probabilité prédite', fontsize=12)
axes[0, 1].set_ylabel('Fréquence', fontsize=12)
axes[0, 1].set_title('Distribution des Probabilités', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[1, 0].plot(fpr, tpr, linewidth=3, label=f'ROC Curve (AUC = {roc_auc:.4f})', color='green')
axes[1, 0].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
axes[1, 0].fill_between(fpr, tpr, alpha=0.2, color='green')
axes[1, 0].set_xlabel('False Positive Rate (1 - Spécificité)', fontsize=12)
axes[1, 0].set_ylabel('True Positive Rate (Sensibilité)', fontsize=12)
axes[1, 0].set_title('Courbe ROC', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, alpha=0.3)

# 4. Métriques récapitulatives
metrics_text = f"""
PERFORMANCE DU MODÈLE

Accuracy:     {accuracy:.4f}
Precision:    {precision:.4f}
Recall:       {recall:.4f}
F1-Score:     {f1:.4f}
ROC-AUC:      {roc_auc:.4f}

MATRICE DE CONFUSION
TN (Vrais Négatifs):   {cm[0, 0]}
FP (Faux Positifs):    {cm[0, 1]}
FN (Faux Négatifs):    {cm[1, 0]}
TP (Vrais Positifs):   {cm[1, 1]}
"""
axes[1, 1].text(0.1, 0.9, metrics_text, transform=axes[1, 1].transAxes,
                fontsize=11, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Analyse des Erreurs et Compréhension du Modèle

In [ ]:
# Identifier les erreurs du modèle
errors = y_pred != y_test
error_indices = np.where(errors)[0]

# Statistiques sur les erreurs
false_negatives = np.where((y_pred == 0) & (y_test == 1))[0]  # Malin prédit comme Bénin
false_positives = np.where((y_pred == 1) & (y_test == 0))[0]  # Bénin prédit comme Malin

print("\n" + "="*60)
print("ANALYSE DES ERREURS")
print("="*60)
print(f"\nNombre total d'erreurs: {len(error_indices)} / {len(y_test)} ({100*len(error_indices)/len(y_test):.2f}%)")
print(f"\nFaux Négatifs (cas malin manqués ❌): {len(false_negatives)}")
print(f"  → Le modèle prédit 'Bénin' mais c'est réellement 'Malin'")
print(f"  → TRÈS GRAVE en contexte médical !")
print(f"\nFaux Positifs (fausses alertes ⚠️): {len(false_positives)}")
print(f"  → Le modèle prédit 'Malin' mais c'est réellement 'Bénin'")
print(f"  → Nécessite investigation supplémentaire")

# Analyser la confiance du modèle sur les erreurs
if len(error_indices) > 0:
    error_confidences = y_pred_proba[error_indices]
    print(f"\nConfiance moyenne sur les erreurs: {error_confidences.mean():.4f}")
    print(f"  → Min: {error_confidences.min():.4f}, Max: {error_confidences.max():.4f}")
    
    # Faux négatifs particulièrement problématiques
    if len(false_negatives) > 0:
        fn_confidences = y_pred_proba[false_negatives]
        print(f"\nConfiance sur les faux négatifs: {fn_confidences.mean():.4f}")
        print(f"  → Le modèle était CERTAIN en se trompant !")

print("\n" + "="*60)

## Structure et Poids du Réseau de Neurones

In [ ]:
# Analyser la structure du réseau
print("\n" + "="*60)
print("STRUCTURE DU RÉSEAU DE NEURONES")
print("="*60)

# Architecture
input_size = X_train_scaled.shape[1]
hidden_sizes = mlp.hidden_layer_sizes
output_size = 1  # Classification binaire

print(f"\n📊 ARCHITECTURE:")
print(f"  Couche Input (0):    {input_size} neurones (features)")
for i, size in enumerate(hidden_sizes, 1):
    print(f"  Couche Hidden ({i}): {size} neurones")
print(f"  Couche Output (L):   {output_size} neurone (probabilité)")

# Paramètres
total_params = 0
print(f"\n🔢 NOMBRE DE PARAMÈTRES:")
print(f"  Input → Hidden(1): {input_size * hidden_sizes[0] + hidden_sizes[0]:,} (poids + biais)")
total_params += input_size * hidden_sizes[0] + hidden_sizes[0]

for i in range(len(hidden_sizes) - 1):
    params = hidden_sizes[i] * hidden_sizes[i+1] + hidden_sizes[i+1]
    print(f"  Hidden({i+1}) → Hidden({i+2}): {params:,}")
    total_params += params

final_params = hidden_sizes[-1] * output_size + output_size
print(f"  Hidden({len(hidden_sizes)}) → Output: {final_params:,}")
total_params += final_params

print(f"\n  TOTAL: {total_params:,} paramètres")

# Analyse des poids de la première couche (input → hidden1)
W_input = mlp.coefs_[0]
print(f"\n📈 POIDS COUCHE INPUT → HIDDEN(1):")
print(f"  Forme: {W_input.shape}")
print(f"  Moyenne: {W_input.mean():.4f}")
print(f"  Écart-type: {W_input.std():.4f}")
print(f"  Min/Max: {W_input.min():.4f} / {W_input.max():.4f}")

# Features les plus influentes (basé sur les poids moyens)
feature_importance = np.abs(W_input).mean(axis=1)
top_features_idx = np.argsort(feature_importance)[-5:][::-1]

print(f"\n⭐ TOP 5 FEATURES INFLUENTES:")
feature_names = data.feature_names
for rank, idx in enumerate(top_features_idx, 1):
    print(f"  {rank}. {feature_names[idx]}: {feature_importance[idx]:.4f}")

print("\n" + "="*60)

In [ ]:
# Visualiser l'importance des features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Top 10 features
top_10_idx = np.argsort(feature_importance)[-10:][::-1]
top_10_importance = feature_importance[top_10_idx]
top_10_names = [data.feature_names[i] for i in top_10_idx]

axes[0].barh(range(len(top_10_names)), top_10_importance, color='steelblue')
axes[0].set_yticks(range(len(top_10_names)))
axes[0].set_yticklabels(top_10_names)
axes[0].set_xlabel('Importance Moyenne (|poids|)', fontsize=12)
axes[0].set_title('Top 10 Features Influentes', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].invert_yaxis()

# 2. Distribution de tous les poids
axes[1].hist(W_input.flatten(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[1].set_xlabel('Valeur des poids', fontsize=12)
axes[1].set_ylabel('Fréquence', fontsize=12)
axes[1].set_title('Distribution des Poids (Input → Hidden1)', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n💡 INTERPRÉTATION:")
print(f"  • Les poids les plus importants correspondent aux features")
print(f"    qui ont le plus d'influence sur la décision du modèle")
print(f"  • Une distribution concentrée autour de 0 indique un")
print(f"    apprentissage équilibré sans sur-ajustement extrême")